# HoneyAI — Model 3: QLoRA LM detector (Colab)
Runtime → Change runtime type → **T4 GPU**. Outputs land in `runs/qlora/<run>/`; download that folder at the end (adapters are ~35 MB per seed for 0.5B).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Either clone (if you've pushed the branch) or upload a zip of the repo folder.

In [ ]:
# Option A: clone (private repo: use a fine-grained read-only token)
# !git clone https://github.com/seoljeong/honeypotttt.git Honeypotttt

# Option B: upload Honeypotttt.zip via the file pane, then:
# !unzip -q Honeypotttt.zip
%cd Honeypotttt

In [ ]:
!pip install -q -r requirements-llm.txt

In [ ]:
# build the v2 simulated dataset (deterministic, ~2 s) if it isn't already in data/v2
import os
if not os.path.exists('data/v2/dataset1_sessions.json'):
    !python scripts/generate_sessions.py


## 2. Sanity floor (CPU, seconds)
The LM has to beat this to support any claim that pretrained-LM semantics help.

In [ ]:
!python scripts/text_baseline.py

## 3. Main run — round 2 (all training strategies), full input, 3 seeds
Defaults: 3 epochs, early stopping, ≤6 prefixes per session (~7.9k training examples). Time per seed on a T4 not yet measured — check the first epoch's timing.

In [ ]:
!python scripts/train_qlora.py

## 4. Experiment runs
Run what you need for the report; each writes its own folder.

In [ ]:
# central ablation: commands only vs commands+responses
!python scripts/train_qlora.py --no-responses

# round 0: train on non-evasive sessions only (round-1 numbers = its evasive-strategy breakdown)
!python scripts/train_qlora.py --round 0

# optional ablations
# !python scripts/train_qlora.py --no-timing
# !python scripts/train_qlora.py --shuffle-steps
# bigger model
# !python scripts/train_qlora.py --model Qwen/Qwen2.5-1.5B --batch-size 4 --grad-accum 4

In [ ]:
import json, glob
for p in sorted(glob.glob('runs/qlora/*/summary.json')):
    s = json.load(open(p, encoding='utf-8')); f = s['full']
    print(p.split('/')[-2], f"full macro-F1 {f['macro_f1']['mean']:.3f}±{f['macro_f1']['std']:.3f}",
          f"AI-evasive recall {f['ai_evasive_recall']['mean']:.3f}", f"p95 {s['latency']['p95_ms']:.0f}ms")

## 5. Online scoring check

In [ ]:
import json, sys, glob; sys.path.insert(0, '.')
from honeyai.models.qlora_cls import OnlineScorer
sc = OnlineScorer(sorted(glob.glob('runs/qlora/*_v2_r2_full/seed42'))[0])
sess = [s for s in json.load(open('data/dataset1_sessions.json', encoding='utf-8')) if s['split']=='test'][-1]
print(sess['actor_label'], sess['evasion_strategy'])
for i in range(1, len(sess['interactions'])+1):
    print(i, sc.score(sess['interactions'][:i]))

In [ ]:
!zip -qr qlora_runs.zip runs/qlora
from google.colab import files; files.download('qlora_runs.zip')